### Just loading data from bigquery (gold layer) and turning into DataFrame for EDA

In [ ]:
# Set up 
from google.cloud import bigquery
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")
client = bigquery.Client(project=PROJECT_ID)

In [30]:
# Query to pull training table from gold 
# Only rows where target exists (excludes last 12 months)
query = f"""
WITH base AS (
    SELECT
        city,
        state,
        tier,
        month,

        -- Optional raw level for diagnostics
        zhvi,

        -- Price features
        zhvi_yoy_smooth,
        zhvi_mom_3m,
        zhvi_volatility_6m,

        -- Rent features
        zori_yoy_smooth,
        zori_mom_3m,
        zori_volatility_6m,

        -- Labor features
        unemployment_rate,
        unemployment_3m_delta,
        jobs_3m_pct,
        jobs_yoy_pct,
        wages_yoy_pct,

        -- Supply features
        permits_yoy_pct,

        -- National macro features
        mortgage_rate_3m_delta,
        mortgage_rate_12m_delta,
        mortgage_rate_volatility_6m,
        cpi_yoy_pct,

        -- Affordability features
        price_to_income_ratio,
        rent_to_income_ratio,

        -- Interaction features
        piti_rate_pressure,
        piti_shock,

        -- Display / diagnostic only
        piti_to_income_ratio,
        mortgage_to_income_ratio,
        monthly_piti,
        monthly_mortgage_payment,
        monthly_property_tax,
        monthly_insurance,

        -- Raw target
        hpa_12m_forward
    FROM `{PROJECT_ID}.housing_gold.gold_market_features`
    WHERE hpa_12m_forward IS NOT NULL
      AND zhvi_yoy_smooth IS NOT NULL
      AND month >= '2019-01-01'
),

national_hpa AS (
    SELECT
        month,
        AVG(hpa_12m_forward) AS national_hpa
    FROM base
    GROUP BY month
)

SELECT
    b.*,
    n.national_hpa,
    ROUND(b.hpa_12m_forward - n.national_hpa, 6) AS hpa_relative
FROM base b
LEFT JOIN national_hpa n
    ON b.month = n.month
ORDER BY b.city, b.month
"""
df_train = client.query(query).to_dataframe(create_bqstorage_client=False)
df_train

,city,state,tier,month,zhvi,zhvi_yoy_smooth,zhvi_mom_3m,zhvi_volatility_6m,zori_yoy_smooth,zori_mom_3m,...,piti_shock,piti_to_income_ratio,mortgage_to_income_ratio,monthly_piti,monthly_mortgage_payment,monthly_property_tax,monthly_insurance,hpa_12m_forward,national_hpa,hpa_relative
0,atlanta,GA,2,2019-01-01,235088.799063,0.0865,0.0068,0.0004,0.0664,0.0063,...,0.064427,0.1790,0.1386,1232.23,954.05,178.28,99.91,0.0540,0.039983,0.014017
1,atlanta,GA,2,2019-02-01,236587.757051,0.0851,0.0068,0.0004,0.0659,0.0050,...,0.001785,0.1785,0.1378,1228.87,948.91,179.41,100.55,0.0551,0.043434,0.011666
2,atlanta,GA,2,2019-03-01,237852.301057,0.0834,0.0062,0.0006,0.0654,0.0040,...,-0.008971,0.1794,0.1386,1235.44,953.98,180.37,101.09,0.0565,0.047111,0.009389
3,atlanta,GA,2,2019-04-01,238851.804031,0.0803,0.0053,0.0011,0.0644,0.0042,...,-0.067169,0.1768,0.1357,1217.06,934.42,181.13,101.51,0.0568,0.049883,0.006917
4,atlanta,GA,2,2019-05-01,239633.983057,0.0769,0.0043,0.0015,0.0635,0.0043,...,-0.091710,0.1764,0.1352,1214.35,930.78,181.72,101.84,0.0550,0.050354,0.004646
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2549,washington_dc,VA,1,2024-09-01,559754.704830,0.0369,0.0015,0.0028,0.0485,0.0037,...,-0.311442,0.3244,0.2791,3238.86,2786.39,251.89,200.58,0.0089,-0.007940,0.016840
2550,washington_dc,VA,1,2024-10-01,561823.278193,0.0358,0.0024,0.0020,0.0473,0.0030,...,-0.360150,0.3366,0.2911,3360.36,2906.22,252.82,201.32,0.0062,-0.008349,0.014549
2551,washington_dc,VA,1,2024-11-01,564120.863474,0.0371,0.0034,0.0013,0.0456,0.0028,...,-0.314250,0.3416,0.2959,3410.15,2954.15,253.85,202.14,0.0038,-0.008300,0.012100
2552,washington_dc,VA,1,2024-12-01,566176.601783,0.0396,0.0038,0.0014,0.0439,0.0027,...,-0.061762,0.3431,0.2973,3425.60,2967.94,254.78,202.88,0.0018,-0.008314,0.010114


In [ ]:
# Make sure month is parsed correctly
df_train["month"] = pd.to_datetime(df_train["month"])
df_train.head()
print(f"Rows: {len(df_train):,}")
print(f"Cities: {df_train['city'].nunique()}")
print(f"Date range: {df_train['month'].min().date()} → {df_train['month'].max().date()}")

# Save parquet
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "training_data.parquet"
df_train.to_parquet(output_path, index=False)

print(f"Saved to {output_path}")

df_train.head()

,city,state,tier,month,zhvi,zhvi_yoy_smooth,zhvi_mom_3m,zhvi_volatility_6m,zori_yoy_smooth,zori_mom_3m,...,piti_shock,piti_to_income_ratio,mortgage_to_income_ratio,monthly_piti,monthly_mortgage_payment,monthly_property_tax,monthly_insurance,hpa_12m_forward,national_hpa,hpa_relative
0,atlanta,GA,2,2019-01-01,235088.799063,0.0865,0.0068,0.0004,0.0664,0.0063,...,0.064427,0.1790,0.1386,1232.23,954.05,178.28,99.91,0.0540,0.039983,0.014017
1,atlanta,GA,2,2019-02-01,236587.757051,0.0851,0.0068,0.0004,0.0659,0.0050,...,0.001785,0.1785,0.1378,1228.87,948.91,179.41,100.55,0.0551,0.043434,0.011666
2,atlanta,GA,2,2019-03-01,237852.301057,0.0834,0.0062,0.0006,0.0654,0.0040,...,-0.008971,0.1794,0.1386,1235.44,953.98,180.37,101.09,0.0565,0.047111,0.009389
3,atlanta,GA,2,2019-04-01,238851.804031,0.0803,0.0053,0.0011,0.0644,0.0042,...,-0.067169,0.1768,0.1357,1217.06,934.42,181.13,101.51,0.0568,0.049883,0.006917
4,atlanta,GA,2,2019-05-01,239633.983057,0.0769,0.0043,0.0015,0.0635,0.0043,...,-0.091710,0.1764,0.1352,1214.35,930.78,181.72,101.84,0.0550,0.050354,0.004646


In [24]:
df.to_parquet("../outputs/training_data.parquet", index=False)
print("Saved to outputs/training_data.parquet")

Saved to outputs/training_data.parquet
